In [ ]:
import pandas as pd

In [ ]:
# Load all 9 datasets
df_customers = pd.read_csv('olist_customers_dataset.csv')
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_items = pd.read_csv('olist_order_items_dataset.csv')
df_payments = pd.read_csv('olist_order_payments_dataset.csv')
df_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')
df_sellers = pd.read_csv('olist_sellers_dataset.csv')
df_geo = pd.read_csv('olist_geolocation_dataset.csv')
df_category_translation = pd.read_csv('product_category_name_translation.csv')

In [ ]:
unique_order_ids = df_orders['order_id'].unique()
print(f"Number of unique order IDs: {len(unique_order_ids)}")

## Merging dataframes

In [ ]:
# Step 1: Join Orders and Customers (Essential for customer_unique_id)
df = pd.merge(df_orders, df_customers, on='customer_id', how='left')

#df = pd.merge(df, geo, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')

# Step 2: Join Items and Products
df = pd.merge(df, df_items, on='order_id', how='left')
df = pd.merge(df, df_products, on='product_id', how='left')

# Step 3: Join Payments and Reviews
df = pd.merge(df, df_payments, on='order_id', how='left')
df = pd.merge(df, df_reviews, on='order_id', how='left')

# Step 4: Join Sellers and Category Translations
df = pd.merge(df, df_sellers, on='seller_id', how='left')
df = pd.merge(df, df_category_translation, on='product_category_name', how='left')

In [ ]:
len(df)

## converting required columns to datetime

In [ ]:
datetime_cols = [
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'order_purchase_timestamp',
]

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col]).dt.floor('us').dt.as_unit('us')

## payment sequental - aggregation

In [ ]:
import pandas as pd
import numpy as np

def aggregate_with_explicit_logic(df_items, df_payments):
    # --- STREAM 1: Items Logic ---
    # Condition: If unique item_ids > 1, SUM. If unique item_ids == 1, FIRST.
    item_stats = df.groupby('order_id').agg({
        'order_item_id': 'nunique',
        'price': ['sum', 'first'],
        'freight_value': ['sum', 'first']
    })
    item_stats.columns = ['unique_items', 'price_sum', 'price_first', 'freight_sum', 'freight_first']
    item_stats = item_stats.reset_index()


    # Apply the Condition
    item_stats['final_price'] = np.where(item_stats['unique_items'] > 1,
                                        item_stats['price_sum'],
                                        item_stats['price_first'])

    item_stats['final_freight'] = np.where(item_stats['unique_items'] > 1,
                                          item_stats['freight_sum'],
                                          item_stats['freight_first'])

    item_stats['unique_items'] = item_stats['unique_items'].replace(0, 1)

    # --- STREAM 2: Payments Logic ---
    # Condition: If unique sequences > 1, SUM. If unique sequences == 1, FIRST.
    pay_stats = df.groupby('order_id').agg({
        'payment_sequential': 'nunique',
        'payment_value': ['sum', 'first'],
        'payment_type': lambda x: ', '.join(sorted(set(map(str, x.dropna().unique()))))
    })
    pay_stats.columns = ['unique_seqs', 'pay_sum', 'pay_first', 'payment_methods']
    pay_stats = pay_stats.reset_index()

    # Apply the Condition
    pay_stats['final_payment_value'] = np.where(pay_stats['unique_seqs'] > 1,
                                               pay_stats['pay_sum'],
                                               pay_stats['pay_first'])

    # --- MASTER MERGE ---
    df_master = item_stats[['order_id', 'unique_items', 'final_price', 'final_freight']].merge(
        pay_stats[['order_id', 'unique_seqs', 'final_payment_value', 'payment_methods']],
        on='order_id',
        how='outer'
    )

    return df_master

# Failproof Checkpoint #1

In [ ]:
df_master = aggregate_with_explicit_logic(df, df)

In [ ]:
df_master.dtypes

## Dropping columns which are already aggregated

In [ ]:
# 1. Define all columns we want to get RID of from the original df
# These are the 'raw' ingredients we've already cooked into master features
cols_to_exclude = [
    'price', 'freight_value', 'order_item_id',
    'payment_value', 'payment_sequential', 'payment_type',
    'review_id', 'review_comment_title',
    'review_comment_message', 'review_creation_date', 'review_answer_timestamp'
]
cols_to_exclude_eda = [
    'price', 'freight_value', 'order_item_id',
    'payment_value', 'payment_sequential', 'payment_type',
]

# 2. Prepare the metadata 'lookup' from the original df
# We drop the excluded columns and then drop_duplicates to get 1 row per order
df_metadata = df.drop(columns=cols_to_exclude, errors='ignore').drop_duplicates(subset='order_id')
df_eda = df.drop(columns=cols_to_exclude_eda, errors='ignore').drop_duplicates(subset='order_id')

# 3. Final Merge
# This joins your clean aggregated features with the timestamps and status
df_final = df_master.merge(df_metadata, on='order_id', how='left')
df_eda = df_master.merge(df_eda, on='order_id', how='left')

In [ ]:
len(df_eda)

In [ ]:
df_eda.to_csv('df_eda.csv', index=False)

# EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/df_eda.csv')

In [ ]:
df.columns

In [ ]:
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])
df['order_delivered_carrier_date'] = pd.to_datetime(df['order_delivered_carrier_date'])
df['shipping_limit_date'] = pd.to_datetime(df['shipping_limit_date'])
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_approved_at'] = pd.to_datetime(df['order_approved_at'])


In [ ]:
df['delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.total_seconds() / (24 * 3600)
df['freight_ratio'] = df['final_freight'] / df['final_price']

In [ ]:
subset = df[['unique_items', 'final_price', 'final_freight']].dropna()

# 1. Unique Items Distribution
plt.figure(figsize=(8, 5))
sns.histplot(subset['unique_items'], bins=20, kde=False, color='navy')
plt.yscale('log')
plt.title('Distribution of Unique Items')
plt.xlabel('Unique Items')
plt.ylabel('Count (Log)')
plt.savefig('dist_unique_items.png')

In [ ]:
# 2. Final Price Distribution
plt.figure(figsize=(8, 5))
sns.histplot(subset['final_price'], bins=50, kde=True, color='navy')
plt.xscale('log')
plt.title('Distribution of Final Price')
plt.xlabel('Final Price (Log)')
plt.ylabel('Frequency')
plt.savefig('dist_final_price.png')

In [ ]:
# 3. Final Freight Distribution
plt.figure(figsize=(8, 5))
sns.histplot(subset['final_freight'], bins=50, kde=True, color='darkblue')
plt.xscale('log')
plt.title('Distribution of Final Freight (Log Scale X)')
plt.xlabel('Final Freight (Log)')
plt.ylabel('Frequency')
plt.savefig('dist_final_freight.png')

In [ ]:

df['product_volume_cm3'] = df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']

    # Drop rows where volume or weight are NaN for plotting
volume_weight_df = df[['product_volume_cm3', 'product_weight_g']].dropna()

    # Plotting Volume Distribution
plt.figure(figsize=(12, 6))
sns.histplot(volume_weight_df['product_volume_cm3'], bins=50, kde=True, color='midnightblue')
plt.xscale('log')
plt.title('Distribution of Product Volume (cm³)')
plt.xlabel('Product Volume (cm³, Log Scale)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(volume_weight_df['product_weight_g'], bins=50, kde=True, color='darkblue')
plt.xscale('log')
plt.title('Distribution of Product Weight (g)')
plt.xlabel('Product Weight (g, Log Scale)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
order_counts = df['order_status'].value_counts()
#sns.set_style("darkgrid")
sns.barplot(x=order_counts.values, y=order_counts.index, palette='Blues', edgecolor="black", linewidth=0.3)

# Use Log Scale if 'delivered' dwarfs the others
plt.xscale('log')
plt.title('Order Status Distribution (Log Scale)')
plt.xlabel('Number of Orders')
plt.ylabel('Status')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=df.dropna(subset=['review_score']), x='review_score', palette='Blues', hue='review_score', legend=False, edgecolor="black", linewidth=0.3)
plt.title('Distribution of Review Scores')
plt.xlabel('Review Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
full_data = df[['unique_items', 'final_price', 'final_freight']].dropna()
full_data = full_data[(full_data['final_price'] > 0) & (full_data['final_freight'] > 0)]

plt.figure(figsize=(12, 8))

# Using np.max ensures the color reflects the actual 1-21 item range
hb = plt.hexbin(
    full_data['final_price'],
    full_data['final_freight'],
    C=full_data['unique_items'],
    reduce_C_function=np.max,
    gridsize=20,
    cmap='plasma',
    xscale='log',
    yscale='log'
)

cb = plt.colorbar(hb)
cb.set_label('Max Unique Items (1-21)')
plt.title('Price vs. Freight')
plt.xlabel('Final Price (Log Scale)')
plt.ylabel('Final Freight (Log Scale)')
plt.savefig('hexbin_full_max_items.png')

In [ ]:
full_data = df[['unique_items', 'final_price', 'final_freight']].dropna()
full_data = full_data[(full_data['final_price'] > 0) & (full_data['final_freight'] > 0)]

plt.figure(figsize=(12, 8))

# Using np.max ensures the color reflects the actual 1-21 item range
hb = plt.hexbin(
    full_data['final_price'],
    full_data['final_freight'],
    C=full_data['unique_items'],
    reduce_C_function=np.max,
    gridsize=600,
    cmap='plasma',
    xscale='log',
    yscale='log'
)

cb = plt.colorbar(hb)
cb.set_label('Max Unique Items (1-21)')
plt.title('Price vs. Freight')
plt.xlabel('Final Price (Log Scale)')
plt.ylabel('Final Freight (Log Scale)')
plt.savefig('hexbin_full_max_items.png')

In [ ]:
# Filter out 'not_defined' payment methods before calculating stats
df_filtered = df[df['payment_methods'] != 'not_defined']

stats = df_filtered.groupby('payment_methods')['final_payment_value'].agg(['count', 'sum']).sort_values(by='count', ascending=False).head(10)

# Plotting with Log Scale for 'Scale' visibility
plt.figure(figsize=(10, 6))
sns.barplot(x=stats['count'], y=stats.index, hue=stats.index, palette='Blues', legend=False, edgecolor="black", linewidth=0.3)
plt.xscale('log')
plt.title('Distribution by Count (Log Scale)')
plt.savefig('payment_method_count_log.png')

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=stats['sum'], y=stats.index, hue=stats.index, palette='Blues', legend=False, edgecolor="black", linewidth=0.3)
plt.xscale('log')
plt.title('Total Payment Value (Log Scale Sum)')
plt.savefig('payment_method_sum_log.png')

In [ ]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_approved_at'] = pd.to_datetime(df['order_approved_at'])

# Calculate delta in hours
time_df = df[['order_purchase_timestamp', 'order_approved_at']].dropna()
time_df['delta_hours'] = (time_df['order_approved_at'] - time_df['order_purchase_timestamp']).dt.total_seconds() / 3600

# Filter for valid deltas and plot
plt.figure(figsize=(10, 6))
sns.histplot(time_df[time_df['delta_hours'] > 0]['delta_hours'], bins=1000,kde=True, color='skyblue', edgecolor='black', linewidth=0.3)
plt.xscale('log')
plt.title('Time from Purchase to Approval (Hours)')
plt.xlabel('Hours (Log Scale)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
delivery_df = df[['order_estimated_delivery_date', 'order_delivered_customer_date']].dropna()
delivery_df['delivery_delta_days'] = (delivery_df['order_estimated_delivery_date'] - delivery_df['order_delivered_customer_date']).dt.total_seconds() / (24 * 3600)

# Plotting the distribution
plt.figure(figsize=(12, 6))
sns.histplot(delivery_df['delivery_delta_days'], bins=100, color='royalblue', kde=True)
plt.axvline(0, color='red', linestyle='--', label='Estimate')
plt.title('Delivery Performance: Days Early vs. Late')
plt.xlabel('Days (Positive = Early)')
plt.xlim(-30, 60) # Focusing on the relevant window
plt.show()

In [ ]:
delivery_df['review_scores'] = df['review_score'].dropna()
def get_status(delta):
    if delta < 0: return 'Late'
    elif delta <= 7: return 'On Time (0-7 days early)'
    else: return 'Early (>7 days early)'

delivery_df['delivery_status'] = delivery_df['delivery_delta_days'].apply(get_status)

# Visualizing Average Review Score by Delivery Status
plt.figure(figsize=(10, 6))
sns.barplot(data=delivery_df, x='delivery_status', y='review_scores', palette='Blues', edgecolor='black', linewidth=0.5)
plt.title('Average Review Score by Delivery Performance')
plt.ylabel('Average Review Score')
plt.ylim(0, 5)
plt.show()

In [ ]:
# --- 1. Code for Top 10 Customer and Seller States (Bar Graphs) ---
# Filter for necessary columns
geo_df = df[['customer_city', 'seller_city']].dropna()
geo_df_state = df[['customer_state', 'seller_state']].dropna()
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Customer States
cust_states = geo_df['customer_city'].value_counts().head(7)
sns.barplot(x=cust_states.values, y=cust_states.index, ax=axes[0], palette='Blues_r', edgecolor='black', linewidth=0.3)
axes[0].set_title('Top 10 Customer Cities')
axes[0].set_xlabel('Number of Orders')

# Seller States
sell_states = geo_df['seller_city'].value_counts().head(7)
sns.barplot(x=sell_states.values, y=sell_states.index, ax=axes[1], palette='coolwarm', edgecolor='black', linewidth=0.3)
axes[1].set_title('Top 10 Seller Cities')
axes[1].set_xlabel('Number of Orders')

plt.tight_layout()
plt.show()

In [ ]:
# --- 2. Code for Intra-State vs. Inter-State Flow (Pie Chart) ---
# Create a boolean column to check if customer and seller are in the same state
geo_df_state['is_same_state'] = geo_df_state['customer_state'] == geo_df_state['seller_state']

# Calculate percentages
state_match = geo_df_state['is_same_state'].value_counts(normalize=True) * 100

plt.figure(figsize=(8, 8))
# Pie chart labels and colors
labels = ['Inter-State (Different)', 'Intra-State (Same)']
colors = ['#608da2', '#c8dfea']

plt.pie(state_match, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors)
plt.title('Logistics Flow: Same State vs. Different State')
plt.show()

In [ ]:
#geo_freight = df_reviews[['customer_state', 'seller_state', 'final_freight']].dropna()
geofreight_mean = geo_freight.groupby(['customer_state', 'seller_state'], as_index=False)['final_freight'].mean()

# Create a boolean column to check if customer and seller are in the same state
geofreight_mean['is_same_state'] = geofreight_mean['customer_state'] == geofreight_mean['seller_state']

# Calculate the mean freight value for each category
average_freight = geofreight_mean.groupby('is_same_state')['final_freight'].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=average_freight, x='is_same_state', y='final_freight', palette='Blues', edgecolor='black', linewidth=0.5, hue='is_same_state', legend=False)

plt.title('Mean Freight Cost: Intra-State vs. Inter-State')
plt.ylabel('Mean Freight Cost ($)')
plt.xlabel('Logistics Type')
plt.xticks([0, 1], ['Inter-State (Different)', 'Intra-State (Same)'])

plt.show()

In [ ]:
if 'product_category_name_english' in df.columns and 'product_photos_qty' in df.columns:
    # Calculate the average product_photos_qty for each category
    avg_photos_per_category = df.groupby('product_category_name_english')['product_photos_qty'].mean().sort_values(ascending=False)

    # Select the top N categories for better readability
    # Using the same number of categories as the previous category distribution plot
    top_N = 11
    top_categories_avg_photos = avg_photos_per_category.head(top_N)

    # Create the bar graph
    plt.figure(figsize=(12, 8))
    sns.barplot(x=top_categories_avg_photos.values, y=top_categories_avg_photos.index, palette='Blues', edgecolor='black',linewidth=0.8, hue=top_categories_avg_photos.index, legend=False)
    plt.title(f'Average Product Photos Quantity for Product Categories')
    plt.xlabel('Average Number of Photos per Product')
    plt.ylabel('Product Category Name (English)')
    plt.tight_layout()
    plt.show()
else:
    print("One or both of the required columns ('product_category_name_english', 'product_photos_qty') not found in DataFrame.")
    print("Available columns:", df.columns.tolist())

In [ ]:
# Plotting Weight vs. Volume with Scatter Plot and Quadrant Coloring
plt.figure(figsize=(12, 8))
# Filter out non-positive weight and volume values for log scale plotting
volume_weight_df_filtered = volume_weight_df[
    (volume_weight_df['product_weight_g'] > 0) &
    (volume_weight_df['product_volume_cm3'] > 0)
].copy()

# Calculate medians for volume and weight to define categories
median_volume = volume_weight_df_filtered['product_volume_cm3'].median()
median_weight = volume_weight_df_filtered['product_weight_g'].median()

# Create a 'quadrant' column based on median values
def assign_quadrant(row):
    vol = row['product_volume_cm3']
    wei = row['product_weight_g']
    if vol <= median_volume and wei <= median_weight:
        return 'Less Volume - Less Weight'
    elif vol > median_volume and wei <= median_weight:
        return 'More Volume - Less Weight'
    elif vol <= median_volume and wei > median_weight:
        return 'Less Volume - More Weight'
    else:
        return 'More Volume - More Weight'

volume_weight_df_filtered['quadrant'] = volume_weight_df_filtered.apply(assign_quadrant, axis=1)

# Calculate counts per quadrant
quadrant_counts = volume_weight_df_filtered['quadrant'].value_counts().to_dict()

# Create the scatter plot with categorical coloring
sns.scatterplot(
    data=volume_weight_df_filtered,
    x='product_volume_cm3',
    y='product_weight_g',
    hue='quadrant',
    palette='cividis',
    alpha=0.6,
    s=15 # point size
)

plt.xscale('log')
plt.yscale('log')
plt.title('Product Weight vs. Volume (by Quadrant)')
plt.xlabel('Product Volume (cm³, Log Scale)')
plt.ylabel('Product Weight (g, Log Scale)')

# Add lines to segregate the quadrants based on medians
plt.axvline(x=median_volume, color='red', linestyle='--', label=f'Median Volume ({median_volume:.0f} cm³)')
plt.axhline(y=median_weight, color='orange', linestyle='--', label=f'Median Weight ({median_weight:.0f} g)')

# Add text labels for the quadrants
# Get current plot limits to place text labels within bounds.
xlims = plt.gca().get_xlim()
ylims = plt.gca().get_ylim()

# Calculate log-transformed midpoints for text placement
log_min_x, log_max_x = np.log10(xlims)
log_min_y, log_max_y = np.log10(ylims)
log_median_volume = np.log10(median_volume)
log_median_weight = np.log10(median_weight)

text_x_lv = 10**(log_min_x + (log_median_volume - log_min_x) / 2) # Midpoint for "Less Volume"
text_x_bv = 10**(log_median_volume + (log_max_x - log_median_volume) / 2) # Midpoint for "Big Volume"
text_y_lw = 10**(log_min_y + (log_median_weight - log_min_y) / 2) # Midpoint for "Less Weight"
text_y_bw = 10**(log_median_weight + (log_max_y - log_median_weight) / 2) # Midpoint for "Big Weight"

plt.text(text_x_lv, text_y_lw, f'Less Volume - Less Weight\nCount: {quadrant_counts.get("Less Volume - Less Weight", 0)}', color='black', ha='center', va='center', fontsize=9, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
plt.text(text_x_bv, text_y_lw, f'More Volume - Less Weight\nCount: {quadrant_counts.get("More Volume - Less Weight", 0)}', color='black', ha='center', va='center', fontsize=9, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
plt.text(text_x_lv, text_y_bw, f'Less Volume - More Weight\nCount: {quadrant_counts.get("Less Volume - More Weight", 0)}', color='black', ha='center', va='center', fontsize=9, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
plt.text(text_x_bv, text_y_bw, f'More Volume - More Weight\nCount: {quadrant_counts.get("More Volume - More Weight", 0)}', color='black', ha='center', va='center', fontsize=9, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# Removed plt.legend() as requested by user
plt.tight_layout()
plt.show()

In [ ]:
status_review = df[['order_status', 'review_score']].dropna()

# Calculate the average review score for each order status
average_review_by_status = status_review.groupby('order_status')['review_score'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=average_review_by_status.index, y=average_review_by_status.values, palette='Blues', edgecolor='black', linewidth=0.3, hue=average_review_by_status.index, legend=False)
plt.title('Average Review Score by Order Status')
plt.xlabel('Order Status')
plt.ylabel('Average Review Score')
plt.ylim(0, 5)
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Ensure the necessary columns exist
if all(col in df.columns for col in ['review_score', 'review_answer_timestamp', 'review_creation_date']):

    # Convert timestamps to datetime objects
    df['review_answer_timestamp'] = pd.to_datetime(df['review_answer_timestamp'])
    df['review_creation_date'] = pd.to_datetime(df['review_creation_date'])

    # Calculate the delta in hours
    review_time_delta_df = df[['review_score', 'review_answer_timestamp', 'review_creation_date']].dropna()
    review_time_delta_df['delta_hours_review'] = (review_time_delta_df['review_answer_timestamp'] - review_time_delta_df['review_creation_date']).dt.total_seconds() / 3600

    # Filter for positive deltas (answer after creation) and reasonable values
    review_time_delta_df = review_time_delta_df[review_time_delta_df['delta_hours_review'] >= 0]

    # Manually add jitter to the x-axis for better visualization of discrete review scores
    jitter_amount = 0.2
    review_time_delta_df['review_score_jittered'] = review_time_delta_df['review_score'] + np.random.normal(0, jitter_amount, size=len(review_time_delta_df))

    # Plotting Review Score vs. Delta between Answer and Creation Timestamp as a box plot
    plt.figure(figsize=(12, 7))
    sns.boxplot(
        data=review_time_delta_df,
        x='review_score',
        y='delta_hours_review',
        hue='review_score', # Explicitly setting hue to address FutureWarning
        palette='cividis',
        linewidth=0.8,
        legend=False
    )
    plt.yscale('log') # Log scale is generally not appropriate for box plots as it distorts distributions
    plt.title('Review Score vs. Time to Answer Review (Days)', fontsize=15)
    plt.xlabel('Review Score')
    plt.ylabel('Time to Answer (Days)')
    plt.show()
else:
    print("One or more required columns ('review_score', 'review_answer_timestamp', 'review_creation_timestamp') not found in DataFrame.")
    print("Available columns:", df.columns.tolist())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure date columns are in datetime format before calculating delta
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'], errors='coerce')
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'], errors='coerce')

# Calculate delivery_delta_days and add it to the main DataFrame 'df'
df['delivery_delta_days'] = (df['order_estimated_delivery_date'] - df['order_delivered_customer_date']).dt.total_seconds() / (24 * 3600)

# Select the relevant columns for correlation analysis
correlation_df = df[['unique_items', 'final_freight', 'final_price', 'product_weight_g', 'review_score', 'delivery_delta_days']].copy()

# Drop rows with any NaN values to ensure accurate correlation calculation
correlation_df.dropna(inplace=True)

# Calculate the correlation matrix
correlation_matrix = correlation_df.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='PuBu', edgecolor='black', linewidth=0.3, fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Key Metrics')
plt.savefig('correlation_map.png')
plt.show()

## Finding 1 particular order with the given condition on payment errors which had status of delivered.

In [ ]:
ghost_trend = df_final[(df_final['final_payment_value'].isna()) & (df_final['order_status'] == 'delivered')]
print(f"Number of Ghost Deliveries: {len(ghost_trend)}")

# Get the order_ids of the ghost trend rows
ghost_order_ids = ghost_trend['order_id'].unique()

# Drop these rows from df_final
df_final = df_final[~df_final['order_id'].isin(ghost_order_ids)]

print(f"Number of rows in df_final after dropping ghost deliveries: {len(df_final)}")

In [ ]:
df_staging = df_final.copy()

In [ ]:
# 4. Final Verification
print(f"Final Table Rows: {len(df_final)}")
print(f"Number of Features: {len(df_final.columns)}")

In [ ]:
df_final = df_staging.copy()

## Checking for null rows

In [ ]:
df_final.isnull().sum()[df_final.isnull().sum() != 0]

# Handling Null values

## final price and final freight

In [ ]:
df_final['final_freight'].describe()

In [ ]:
median_final_freight = df_final['final_freight'].median()

# Condition 1: final_price and final_freight are null, final_payment_value is NOT null
mask_cond1 = (df_final['final_price'].isnull()) & \
             (df_final['final_freight'].isnull()) & \
             (df_final['final_payment_value'].notnull()) & \
             (df_final['final_payment_value'] != 0)

# Apply updates for Condition 1
df_final.loc[mask_cond1, 'final_freight'] = median_final_freight
df_final.loc[mask_cond1, 'final_price'] = df_final.loc[mask_cond1, 'final_payment_value'] - median_final_freight

# Condition 2: final_price and final_freight are null, final_payment_value is 0
mask_cond2 = (df_final['final_price'].isnull()) & \
             (df_final['final_freight'].isnull()) & \
             (df_final['final_payment_value'] == 0)

# Apply updates for Condition 2
df_final.loc[mask_cond2, 'final_price'] = 0
df_final.loc[mask_cond2, 'final_freight'] = 0

print(f"Nulls remaining in final_price: {df_final['final_price'].isnull().sum()}")
print(f"Nulls remaining in final_freight: {df_final['final_freight'].isnull().sum()}")

## order_approved_at

In [ ]:
df_final['approval_delta'] = (df_final['order_approved_at'] - df_final['order_purchase_timestamp']).dt.total_seconds() / (60 * 60) #hours

# Calculate the average approval delta for each payment method and order status
approval_delta_by_payment = df_final.groupby(['payment_methods', 'order_status'])['approval_delta'].mean().reset_index()

In [ ]:
# Condition 1: order status - delivered, payment methods - boleto
cond1_delta = approval_delta_by_payment[
    (approval_delta_by_payment['order_status'] == 'delivered') &
    (approval_delta_by_payment['payment_methods'] == 'boleto')
]
print("1. Average delta for 'delivered' status and 'boleto' payment methods:")
display(cond1_delta[['payment_methods', 'order_status', 'approval_delta']])

# Condition 2: order status - cancelled, payment methods - boleto, credit_card, [credit_card, voucher], not_defined, voucher
cond2_payment_methods = ['boleto', 'credit_card', 'credit_card, voucher', 'not_defined', 'voucher']
cond2_delta = approval_delta_by_payment[
    (approval_delta_by_payment['order_status'] == 'canceled') &
    (approval_delta_by_payment['payment_methods'].isin(cond2_payment_methods))
]
print("\n2. Average delta for 'canceled' status and specified payment methods:")
display(cond2_delta[['payment_methods', 'order_status', 'approval_delta']])

# Condition 3: order status - created, payment methods - boleto, credit_card
cond3_payment_methods = ['boleto', 'credit_card']
cond3_delta = approval_delta_by_payment[
    (approval_delta_by_payment['order_status'] == 'created') &
    (approval_delta_by_payment['payment_methods'].isin(cond3_payment_methods))
]
print("\n3. Average delta for 'created' status and specified payment methods:")
display(cond3_delta[['payment_methods', 'order_status', 'approval_delta']])

In [ ]:
import pandas as pd

print(f"Nulls remaining in 'order_approved_at' before imputation: {df_final['order_approved_at'].isnull().sum()}")

# Create a mapping dictionary for average approval deltas
# 'approval_delta_by_payment' DataFrame should already be available from previous steps
delta_mapping = approval_delta_by_payment.set_index(['payment_methods', 'order_status'])['approval_delta'].to_dict()

# Identify rows where order_approved_at is null
null_approved_at_mask = df_final['order_approved_at'].isnull()

# Calculate estimated approval delta (in hours) for these nulls
# Using .get with default None for cases where a combination might not exist in the mapping
df_final.loc[null_approved_at_mask, 'estimated_approval_hours'] = df_final[null_approved_at_mask].apply(
    lambda row: delta_mapping.get((row['payment_methods'], row['order_status'])), axis=1
)

# Convert estimated hours to timedelta
estimated_timedelta = pd.to_timedelta(df_final['estimated_approval_hours'], unit='h').dt.floor('us').dt.as_unit('us')

# Fill nulls in 'order_approved_at' by adding estimated_timedelta to 'order_purchase_timestamp'
df_final.loc[null_approved_at_mask, 'order_approved_at'] = \
    df_final.loc[null_approved_at_mask, 'order_purchase_timestamp'] + estimated_timedelta[null_approved_at_mask]

# Drop the temporary estimated_approval_hours column
df_final.drop(columns=['estimated_approval_hours'], errors='ignore', inplace=True)

# Verify nulls remaining in 'order_approved_at'
print(f"Nulls remaining in 'order_approved_at' after imputation: {df_final['order_approved_at'].isnull().sum()}")


## dealing with these null discrepencies

In [ ]:
# 1. Define the masks for the three 'No-Average' categories
mask_not_defined_cancel = (df_final['payment_methods'] == 'not_defined') & (df_final['order_status'] == 'canceled')
mask_boleto_created = (df_final['payment_methods'] == 'boleto') & (df_final['order_status'] == 'created')
mask_cc_created = (df_final['payment_methods'] == 'credit_card') & (df_final['order_status'] == 'created')

# 2. Apply the 'Neutral Fill' (Lag = 0)
# We use the purchase timestamp so the model sees these as 'instant' events
target_mask = mask_not_defined_cancel | mask_boleto_created | mask_cc_created
df_final.loc[target_mask, 'order_approved_at'] = df_final.loc[target_mask, 'order_purchase_timestamp']

# 3. Create a 'Failure Signal' Feature
# This is CRITICAL. It tells the Logistic Regression that this 0-lag
# isn't a 'fast success,' it's a 'fast failure.'
df_final['is_process_failure'] = target_mask.astype(int)

print(f"Repaired {target_mask.sum()} rows in the 'Dead Zone'.")

## order_delivered_carrier_date

In [ ]:
# Check the status of rows where carrier_date is null
carrier_null_status = df_final[df_final['order_delivered_carrier_date'].isna()]['order_status'].value_counts()
total_null_carrier_status = df_final['order_delivered_carrier_date'].isnull().sum()
print("Statuses for null carrier dates:")
print("Total null for carrier date: ",total_null_carrier_status )
print(carrier_null_status)

In [ ]:
#stuck_statuses = ['unavailable','canceled','invoiced','processing','created','approved','delivered']
df_final['review_score'] = df_final['review_score'].fillna(3.0)
# 2. Identify Ghost Successes
stuck_ghost_mask =  (df_final['order_delivered_carrier_date'].isna()) & \
                    (df_final['review_score'] >= 4)

# 3. Identify True Failures (Stuck forever)
stuck_failure_mask = (df_final['order_delivered_carrier_date'].isna()) & \
                      ((df_final['review_score'] < 4) | (df_final['review_score'].isna()))

median_proc = (df_final['order_delivered_carrier_date'] - df_final['order_approved_at']).median()


# 4. Apply the split (Using the median_proc from earlier)
df_final.loc[stuck_ghost_mask, 'order_delivered_carrier_date'] = \
    df_final.loc[stuck_ghost_mask, 'order_approved_at'] + median_proc

df_final.loc[stuck_failure_mask, 'order_delivered_carrier_date'] = \
    df_final.loc[stuck_failure_mask, 'order_approved_at']

print(f"Stuck Orders - Repaired {stuck_ghost_mask.sum()} Ghosts and {stuck_failure_mask.sum()} Failures.")

## order_delivered_customer_date

In [ ]:
# 1. Identify Ghost Successes (System says null, Customer says 5-stars)
# We treat these as 'on-time' to avoid skewing the model with fake delays.
delivery_ghost_mask = (df_final['order_delivered_customer_date'].isna()) & \
                      (df_final['review_score'] >= 4)

df_final.loc[delivery_ghost_mask, 'order_delivered_customer_date'] = \
    df_final.loc[delivery_ghost_mask, 'order_estimated_delivery_date']

# 2. Identify True Failures (The 1,700+ rows that never arrived)
# We apply a 30-day penalty to make these 'loud' outliers for the model.
delivery_fail_mask = (df_final['order_delivered_customer_date'].isna())

df_final.loc[delivery_fail_mask, 'order_delivered_customer_date'] = \
    df_final.loc[delivery_fail_mask, 'order_estimated_delivery_date'] + pd.to_timedelta(30, unit='D')

print(f"Repaired {delivery_ghost_mask.sum()} Ghost Successes and {delivery_fail_mask.sum()} True Failures.")

# product_id, price, freight_value, product_weight_g, product_description_length, product_photos_qty being null

In [ ]:
# Check the review distribution of the rows with missing IDs
null_id_reviews = df_final[df_final['product_id'].isna()]['review_score'].value_counts(normalize=True)
print("Review Distribution for Missing IDs:")
print(null_id_reviews)

In [ ]:
# 1. Create the 'Failure Flag' first (This is one of your 31 features!)
#df_final['is_orphaned_order'] = df_final['product_id'].isna().astype(int)

# 2. Fill the ID with a placeholder
# (Logistic Regression doesn't use the ID, but your code needs it to not be NaN)
df_final['product_id'] = df_final['product_id'].fillna('orphaned_product')

# 3. Fill Categorical features (for One-Hot Encoding)
#df_final['product_category_name_english'] = df_final['product_category_name_english'].fillna('unknown')

# 4. Fill Numerical features with the MEDIAN
# This makes these features 'neutral' (0 after scaling) for these rows
num_product_features = [
     'product_weight_g',
    'product_description_lenght', 'product_photos_qty'
]

for col in num_product_features:
    if col in df_final.columns:
        df_final[col] = df_final[col].fillna(df_final[col].median())

#print(f"Product IDs cleaned. {df_final['is_orphaned_order'].sum()} orphaned rows preserved.")

## seller_id, seller_city, seller_state, shipping_limit_date, seller_zip_code_prefix

In [ ]:
# 1. Handle the String Identifiers and Categories
seller_str_cols = ['seller_id', 'seller_city', 'seller_state']
for col in seller_str_cols:
    if col in df_final.columns:
        df_final[col] = df_final[col].fillna('unknown_seller_info')

# 2. Handle the Shipping Limit Date
# We sync it to the approval date to create a 'neutral' shipping window
df_final['shipping_limit_date'] = df_final['shipping_limit_date'].fillna(df_final['order_approved_at'])

# 3. Handle the Numerical Zip Code (if you're keeping it)
if 'seller_zip_code_prefix' in df_final.columns:
    df_final['seller_zip_code_prefix'] = df_final['seller_zip_code_prefix'].fillna(0)

print(f"Seller Cleanup Complete: {len(df_final[df_final['seller_id'] == 'unknown_seller_info'])} rows neutralized.")

In [ ]:
# 1. Product Text & Category (2,191 nulls)
df_final['product_category_name'] = df_final['product_category_name'].fillna('unknown')

# 2. Product Dimensions & Lengths (791 - 2,191 nulls)
# We use a loop for the numeric product specs
prod_specs = [
    'product_name_lenght',
    'product_length_cm', 'product_height_cm', 'product_width_cm'
]

for col in prod_specs:
    if col in df_final.columns:
        df_final[col] = df_final[col].fillna(df_final[col].median())

In [ ]:
# 1. Fix the single payment installment (Mode or Median)
df_final['payment_installments'] = df_final['payment_installments'].fillna(1)

# 2. Recalculate the Approval Delta
# This ensures those 160 rows finally have a numeric value for the model
df_final['approval_delta'] = (df_final['order_approved_at'] -
                              df_final['order_purchase_timestamp']).dt.total_seconds() / 3600

In [ ]:
# 1. Convert to datetime if you haven't already
df_reviews['review_answer_timestamp'] = pd.to_datetime(df_reviews['review_answer_timestamp'])

# 2. Sort by the response time (Newest at the bottom)
df_reviews_sorted = df_reviews.sort_values(by='review_answer_timestamp', ascending=True)

# 3. Drop duplicates on order_id, keeping the latest response
df_reviews_unique = df_reviews_sorted.drop_duplicates(subset='order_id', keep='last')

df_reviews_unique.drop('review_score', axis=1, inplace=True)
# Verification
print(f"Unique Reviews: {len(df_reviews_unique)}") # Should be 98,673



In [ ]:
df_final = pd.merge(df_final, df_reviews_unique, on='order_id', how='left')

In [ ]:
# Label the missing IDs so they are no longer float NaNs
df_final['review_id'] = df_final['review_id'].fillna('no_review_record')

cols_to_fix = {
    'review_comment_message': 'no_message',
    'review_comment_title': 'no_title'
}

for col, placeholder in cols_to_fix.items():
    # Force to string
    df_final[col] = df_final[col].astype(str)
    # Replace literal 'nan', empty strings, or any amount of whitespace
    df_final[col] = df_final[col].replace(['nan', 'NaN', 'None', ''], placeholder)
    df_final[col] = df_final[col].replace(r'^[^a-zA-Z0-9]*$', placeholder, regex=True)

# If review dates are missing, set them equal to the order purchase timestamp
# This ensures any 'response_lag' calculation results in 0
df_final['review_creation_date'] = df_final['review_creation_date'].fillna(df_final['order_purchase_timestamp'])
df_final['review_answer_timestamp'] = df_final['review_answer_timestamp'].fillna(df_final['order_purchase_timestamp'])

review_cols = [
    'review_score', 'review_id', 'review_comment_title',
    'review_comment_message', 'review_creation_date', 'review_answer_timestamp'
]

print("Remaining Nulls in Review Section:")
print(df_final[review_cols].isna().sum())

In [ ]:
empty_message_reviews = df_final[df_final['review_comment_title'].str.strip().eq('')]
print("Reviews where 'review_comment_message' is empty or contains only spaces:")
display(len(empty_message_reviews))

In [ ]:
df_final['product_category_name_english'] = df_final['product_category_name_english'].fillna('unknown')
print(f"Nulls remaining in 'product_category_name_english': {df_final['product_category_name_english'].isnull().sum()}")

In [ ]:
df_final.isnull().sum()[df_final.isnull().sum() != 0]

In [ ]:
df_final.columns

In [ ]:
len(df_final)

# FailProof checkpoint #2

In [ ]:
df_staging = df_final.copy()

In [ ]:
df_final = df_staging.copy()

## bert integration for review scores

In [ ]:
# Create the 'full_review_text' column
def final_bert_prep(row):
    t = str(row['review_comment_title']).strip()
    m = str(row['review_comment_message']).strip()

    # Handle the placeholders
    t_clean = "" if t in ['no_title', 'nan', 'None'] else t
    m_clean = "" if m in ['no_message', 'nan', 'None'] else m

    # Join Title and Message with a period
    combined = f"{t_clean}. {m_clean}".strip(". ")
    return combined if combined != "" else "SKIP_AND_NEUTRALIZE"

# 1. Apply the function to your main dataframe
df_final['full_review_text'] = df_final.apply(final_bert_prep, axis=1)

# 2. Export it so you can upload it to Google Colab or your GPU environment
#df_final.to_csv('df_final_bert_ready.csv', index=False)

In [ ]:
# 1. Drop the original source columns
df_final = df_final.drop(columns=['review_comment_title', 'review_comment_message'])

print("Redundant text columns dropped. Memory usage optimized.")

In [ ]:
# Select only what the AI needs to see
cols_for_ai = ['order_id', 'review_id', 'full_review_text']

# Create the minimal inference file
df_inference = df_final[df_final['full_review_text'] != 'SKIP_AND_NEUTRALIZE'][cols_for_ai]

# Save this small file (this is your 'df_final_bert_ready')
df_inference.to_csv('inference_input.csv', index=False)

print(f"Minimal file created! Rows to process: {len(df_inference)}")

In [ ]:
bert_results = pd.read_csv('bert_comparison_results.csv')

In [ ]:
len(bert_results)

In [ ]:
df_final.columns

# FailProof Checkpoint #3

In [ ]:
df_reviews = df_final.copy()

In [ ]:
df_reviews = df_final.merge(
    bert_results[['order_id', 'ugues_neg_prob']],
    on='order_id',
    how='left'
)

In [ ]:
df_reviews['ugues_neg_prob'].isna().sum()

In [ ]:
len(df_reviews)

In [ ]:
df_reviews.columns

In [ ]:
bert_repr = df_reviews.copy()

In [ ]:
df_reviews['ugues_neg_prob'] = df_reviews['ugues_neg_prob'].fillna(0.5)

In [ ]:
# 1. Transform stars into friction (1.0 = bad, 0.0 = good)
df_reviews['stars_fric'] = (5 - df_reviews['review_score']) / 4

# 2. Average with BERT Negative Probability
df_reviews['consensus_friction'] = (df_reviews['ugues_neg_prob'] + df_reviews['stars_fric']) / 2
# Calculate the absolute difference between the two signals
df_reviews['friction_dissonance'] = abs(df_reviews['ugues_neg_prob'] - df_reviews['stars_fric'])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Create a label to distinguish human voices from imputed values
# Based on your files, 'SKIP_AND_NEUTRALIZE' was the placeholder for no review
df_reviews['Source'] = df_reviews['full_review_text'].apply(
    lambda x: 'Imputed Neutral (No Review)' if x == 'SKIP_AND_NEUTRALIZE' else 'BERT Analyzed (Actual Text)'
)

plt.figure(figsize=(12, 7))

# 2. Use a stacked histogram to show the 57k sitting on top of the 42k
sns.histplot(
    data=df_reviews,
    x='ugues_neg_prob',
    hue='Source',
    multiple="stack",
    palette={'BERT Analyzed (Actual Text)': '#1abc9c', 'Imputed Neutral (No Review)': '#95a5a6'},
    bins=50,
    edgecolor='white',
    linewidth=0.5
)

# 3. Formatting for clarity
plt.title('Distribution of BERT Negativity Scores: The Impact of Missing Reviews', fontsize=15)
plt.xlabel('Negativity Probability (0 = Positive, 1 = Negative)', fontsize=12)
plt.ylabel('Number of Orders', fontsize=12)

# Annotation for your thesis
plt.annotate('57k orders without reviews\nforced to neutral 0.5',
             xy=(0.5, 57000), xytext=(0.6, 50000),
             arrowprops=dict(facecolor='black', shrink=0.05, width=2),
             fontsize=11, color='#7f8c8d', fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.yscale('log')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

# We look at the distribution of BERT Negative Prob for each Star Rating
sns.violinplot(x='review_score', y='ugues_neg_prob', data=df_reviews, palette='RdYlGn_r',edgecolor='black', linewidth=0.6)
plt.title('Semantic Calibration: BERT Negativity vs. Star Rating', fontsize=14)
plt.xlabel('Customer Star Rating')
plt.ylabel('BERT Negative Probability')
plt.axhline(0.5, color='red', linestyle='--', alpha=0.6, label='Friction Threshold')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df_reviews['consensus_friction'], bins=50, kde=True, color='teal', edgecolor='white',linewidth=0.5)
plt.title('Distribution of Consensus Friction Scores', fontsize=14)
plt.xlabel('Negativity Probability (0 = Positive, 1 = Negative)')
plt.ylabel('Count of Reviews')
plt.yscale('log')
plt.axvline(0.5, color='red', linestyle='--', label='Sentiment Threshold')
plt.legend()
plt.show()

# Grouping to customer unique ids for logistical user churn prediction

In [ ]:
import pandas as pd
import numpy as np

# 1. PREPARE DATETIMES
df_reviews['order_purchase_timestamp'] = pd.to_datetime(df_reviews['order_purchase_timestamp'])
df_reviews['order_delivered_customer_date'] = pd.to_datetime(df_reviews['order_delivered_customer_date'])
df_reviews['order_estimated_delivery_date'] = pd.to_datetime(df_reviews['order_estimated_delivery_date'])

# 2. SORT TO IDENTIFY LATEST ORDER
# We need the most recent experience for the "Final Impression"
df_reviews = df_reviews.sort_values(['customer_unique_id', 'order_purchase_timestamp'], ascending=[True, False])

# 3. COLLAPSE TO CUSTOMER LEVEL
# We apply your matrix logic here
customer_matrix = df_reviews.groupby('customer_unique_id').agg(
    order_count=('order_id', 'count'),
    fail_semantic=('consensus_friction', lambda x: (x.iloc[0] > 0.5).astype(int)),
    #consensus_friction=('consensus_friction', 'first'),
    # Logistics Fail: Only from the latest order
    last_purchase_date=('order_purchase_timestamp', 'max'),
    #latest_estimated=('order_estimated_delivery_date', 'first'),
    #latest_delivered=('order_delivered_customer_date', 'first'),
    #fail_process=('order_status', lambda x: x.isin(['canceled', 'unavailable']).max().astype(int)),
).reset_index()

snapshot_date = df_reviews['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

customer_matrix['days_since_last_order'] = (
    snapshot_date - customer_matrix['last_purchase_date']
).dt.days
'''
customer_matrix['delivery_delay_delta'] = (
    customer_matrix['latest_estimated'] - customer_matrix['latest_delivered']
).dt.total_seconds() / (3600 * 24)
'''
# 4. DEFINE INACTIVITY (180 DAY CUTOFF)
dataset_today = customer_matrix['last_purchase_date'].max()
customer_matrix['is_inactive'] = (
    (dataset_today - customer_matrix['last_purchase_date']).dt.days > 180
).astype(int)

# 1. Operational Pillar: Failure to deliver on time
# (Using 0 as threshold because positive delta means early/on-time)
#customer_matrix['fail_logistics'] = (customer_matrix['delivery_delay_delta'] < 0).astype(int)


# Target variable creation using Bricolage process

In [ ]:
# 5. THE FINAL CHURN TARGET (y)
# Based on your fixed matrix: Inactive AND Semantic Friction
# (Because our fail_process and fail_late_delivery are now semantic-gated)

customer_matrix['target_churn'] = (
    (customer_matrix['is_inactive'] == 1) &
    (customer_matrix['fail_semantic'] == 1)
).astype(int)


'''
# A customer is a churner IF:
# They are Inactive AND (They had a Semantic fail OR Logistics fail OR Process fail)
customer_matrix['target_churn'] = (
    (customer_matrix['is_inactive'] == 1) &
    (
        (customer_matrix['fail_semantic'] == 1) |
        (customer_matrix['fail_logistics'] == 1)
    )
).astype(int)
'''

In [ ]:
churn_distributions = customer_matrix['target_churn'].value_counts(normalize=True)
print("Churn Distribution (0=No Churn, 1=Churn):")
print(churn_distributions)

In [ ]:
print(customer_matrix['target_churn'].value_counts())

In [ ]:
len(customer_matrix)

In [ ]:
customer_matrix.to_csv('customer_matrix.csv', index=True)

In [ ]:
df_reviews.to_csv('df_reviews.csv', index=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Data Setup
labels = ['Validity', 'Simplicity', 'Predictability', 'Portability', 'Resource\nEfficiency']
stats = [4, 5, 4, 3, 5]
num_vars = len(labels)

# Split the circle into even parts and close the loop
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
stats += stats[:1]
angles += angles[:1]

# 2. Set the Dark Theme Environment
plt.rcParams.update({
    "figure.facecolor": "white", # Dark outer background
    "axes.facecolor": "white",   # Dark inner background
    "text.color": "black",
    "axes.labelcolor": "black"
})

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

# 3. Aesthetics & Grid
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Draw axes and labels
plt.xticks(angles[:-1], labels, color='#E0E0E0', size=13, fontweight='bold')

# Set y-axis (concentric circles)
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="#888888", size=10)
plt.ylim(0, 5)

# Customize the grid lines to be subtle
ax.grid(color='#444444', linestyle='--', linewidth=0.8)
ax.spines['polar'].set_visible(False) # Hide the outer circle spine

# 4. Plotting the "Glowing" Data
# Using a bright Cyan/Electric Blue (#00E5FF)
ax.plot(angles, stats, color='#00E5FF', linewidth=3, marker='o',
        markersize=10, markerfacecolor='#00E5FF', markeredgecolor='white')

# Fill with a gradient effect (using alpha)
ax.fill(angles, stats, color='#00E5FF', alpha=0.25)

# 6. Titles
plt.title('Bricolage Validation: "Strict AND" Formulation',
          size=22, color='white', pad=40, fontweight='bold')
plt.figtext(0.5, 0.05, 'Score based on Framework Criteria (Guerdan et al., 2025)',
            ha='center', color='#AAAAAA', size=11, style='italic')

plt.tight_layout()
plt.savefig('radar_plot_improved.png', transparent=False)
plt.show()

In [ ]:
df_reviews.columns

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. DYNAMIC TODAY (No hardcoding)
dataset_today = pd.to_datetime(df_orders['order_purchase_timestamp']).max()

# 2. DATA PREP (Raw Scraps)
customer_matrix['recency_days'] = (dataset_today - pd.to_datetime(customer_matrix['last_purchase_date'])).dt.days
X_raw = customer_matrix[['consensus_friction', 'recency_days']]

# 3. MANUAL LOGIC (Target Definition)
y_manual = ((customer_matrix['recency_days'] > 180) &
            (customer_matrix['consensus_friction'] > 0.5)).astype(int)

# 4. K-MEANS TRIANGULATION
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

# Align labels (Ensure 1 = Churn)
if (kmeans_labels == y_manual.values).mean() < 0.5:
    kmeans_labels = 1 - kmeans_labels

# 5. FINAL COMPARISON DATA
print("--- Methodological Triangulation: Consensus Report ---")
print(classification_report(y_manual, kmeans_labels))

# Logistic Regression

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict

# 1. Load the Datasets
cust_matrix = pd.read_csv('customer_matrix.csv')
order_data = pd.read_csv('df_reviews.csv')

# 2. Feature Engineering on Order-level Data
# Date conversions
order_data['order_estimated_delivery_date'] = pd.to_datetime(order_data['order_estimated_delivery_date'], errors='coerce')
order_data['order_delivered_customer_date'] = pd.to_datetime(order_data['order_delivered_customer_date'], errors='coerce')
order_data['shipping_limit_date'] = pd.to_datetime(order_data['shipping_limit_date'], errors='coerce')
order_data['order_delivered_carrier_date'] = pd.to_datetime(order_data['order_delivered_carrier_date'], errors='coerce')

# Numerical deltas (in days): Positive = Early/On-time, Negative = Delayed
order_data['delivery_delay_delta'] = (order_data['order_estimated_delivery_date'] - order_data['order_delivered_customer_date']).dt.total_seconds() / (3600*24)
order_data['carrier_delivery_delay_delta'] = (order_data['shipping_limit_date'] - order_data['order_delivered_carrier_date']).dt.total_seconds() / (3600*24)

# Product Physical Descriptions (Volume)
order_data['product_volume'] = order_data['product_length_cm'] * order_data['product_height_cm'] * order_data['product_width_cm']

# New feature: is_same_state
order_data['is_same_state'] = (order_data['customer_state'] == order_data['seller_state']).astype(int)

'''
########################################################################################################################
# 3. Aggregation to Customer Level
num_features = ['final_price', 'final_freight', 'unique_items', 'delivery_delay_delta',
                'carrier_delivery_delay_delta', 'product_weight_g', 'product_volume', 'payment_installments', 'is_same_state']
cust_num = order_data.groupby('customer_unique_id')[num_features].mean().reset_index()

cat_features = ['payment_methods', 'order_status']
cust_cat = order_data.groupby('customer_unique_id')[cat_features].first().reset_index()






cust_combined = pd.merge(cust_num, cust_cat, on='customer_unique_id')

# 4. Merge with Target Variable (Churn)
final_df = pd.merge(cust_combined, cust_matrix[['customer_unique_id', 'target_churn']], on='customer_unique_id').dropna()

# Reset index after dropna to ensure a clean, contiguous index
final_df = final_df.reset_index(drop=True)

# 5. Model Preparation & Training
X = pd.get_dummies(final_df.drop(columns=['customer_unique_id', 'target_churn']), columns=cat_features, drop_first=True)
y = final_df['target_churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

# Train Model with balanced class weights
model = LogisticRegression(max_iter=3000, class_weight='balanced')
model.fit(X_train, y_train)

# 6. Results
print(f"Accuracy: {accuracy_score(y_test, model.predict(X_test))}")
print(classification_report(y_test, model.predict(X_test)))
'''

In [ ]:
# 3. Aggregation to Customer Level
# Add 'is_same_state' to the numerical features list
num_features = [
    'final_price', 'final_freight', 'unique_items', 'delivery_delay_delta',
    'carrier_delivery_delay_delta', 'product_weight_g', 'product_volume',
    'payment_installments', 'is_same_state'
]

cust_num = order_data.groupby('customer_unique_id')[num_features].mean().reset_index()

cat_features = ['payment_methods','order_status']
cust_cat = order_data.groupby('customer_unique_id')[cat_features].first().reset_index()

cust_combined = pd.merge(cust_num, cust_cat, on='customer_unique_id')

# 4. Merge with Target Variable
final_df = pd.merge(cust_combined, cust_matrix[['customer_unique_id', 'target_churn']], on='customer_unique_id').dropna()

In [ ]:
# 5. Pipeline Preparation
# IMPORTANT: Use final_df directly. Do NOT use pd.get_dummies here.
X = final_df.drop(columns=['customer_unique_id', 'target_churn'])
y = final_df['target_churn']

# The Pipeline automatically handles OneHotEncoding for cat_features
# and Scaling for num_features without destroying column names.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
    ])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42))
])

# 6. Stratified Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(pipeline, X, y, cv=skf, scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], n_jobs=-1, return_estimator=True)


In [ ]:
# 3. Create a Detailed Performance Table
cv_df = pd.DataFrame(cv_results)

# Rename columns from cv_results keys to desired display names
cv_df = cv_df.rename(columns={
    'fit_time': 'Fit Time',
    'score_time': 'Score Time',
    'test_accuracy': 'Accuracy',
    'test_precision': 'Precision',
    'test_recall': 'Recall',
    'test_f1': 'F1 Score',
    'test_roc_auc': 'ROC-AUC'
})

# Set index
cv_df.index = [f'Fold {i+1}' for i in range(len(cv_df))]

# Metrics for which we want mean/std
metrics_to_summarize = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'Fit Time', 'Score Time']

summary_stats_data = {}
for metric in metrics_to_summarize:
    summary_stats_data[metric] = [cv_df[metric].mean(), cv_df[metric].std()]

summary_stats = pd.DataFrame(summary_stats_data, index=['MEAN', 'STD DEV'])

# Select desired columns from cv_df for final report (excluding 'estimator')
display_cols = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'Fit Time', 'Score Time']
final_report_table = pd.concat([cv_df[display_cols], summary_stats])

print(final_report_table.round(4))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import StratifiedKFold

# 1. Setup for plotting
fig, ax = plt.subplots(figsize=(8, 8))

tprs = []
aucs = []
mean_fpr = np.linspace(0, 1, 100)

# 2. Iterate through the folds to calculate curves
# Assuming 'pipeline' is your Logistic Regression pipeline and X, y are prepared
for i, (train, test) in enumerate(skf.split(X, y)):
    pipeline.fit(X.iloc[train], y.iloc[train])

    # Get probabilities for the positive class (Churn)
    y_probas = pipeline.predict_proba(X.iloc[test])[:, 1]

    # Compute ROC curve and area
    fpr, tpr, thresholds = roc_curve(y.iloc[test], y_probas)
    roc_auc = auc(fpr, tpr)
    aucs.append(roc_auc)

    # Interpolate to common FPR for averaging
    interp_tpr = np.interp(mean_fpr, fpr, tpr)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)

    # Plot individual fold
    ax.plot(fpr, tpr, alpha=0.3, lw=1, label=f'ROC Fold {i+1} (AUC = {roc_auc:.2f})')

# 3. Plot the Mean ROC
mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = auc(mean_fpr, mean_tpr)
std_auc = np.std(aucs)

ax.plot(mean_fpr, mean_tpr, color='b',
        label=f'Mean ROC (AUC = {mean_auc:.2f} $\pm$ {std_auc:.2f})',
        lw=2, alpha=.8)

# 4. Plot Chance (Random Guessing)
ax.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r', label='Chance', alpha=.8)

# 5. Styling for the Presentation
ax.set(xlim=[-0.05, 1.05], ylim=[-0.05, 1.05],
       title="ROC Curve: Validating 'Confirmed Churn' Predictability",
       xlabel="False Positive Rate",
       ylabel="True Positive Rate")
ax.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract mean and standard deviation for the relevant metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
mean_scores = final_report_table.loc['MEAN', metrics]
std_dev_scores = final_report_table.loc['STD DEV', metrics]

plt.figure(figsize=(10, 6))
sns.barplot(x=mean_scores.index, y=mean_scores.values, palette='magma', hue=mean_scores.index, legend=False)

# Add error bars
plt.errorbar(x=mean_scores.index, y=mean_scores.values, yerr=std_dev_scores.values,
             fmt='none', capsize=5, color='black')

plt.title('Mean Model Performance Metrics Across 5-Fold Cross-Validation', fontsize=14)
plt.xlabel('Metric', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.ylim(0, 1) # Metrics are typically between 0 and 1
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Annotate bars with their mean values
for index, value in enumerate(mean_scores.values):
    plt.text(index, value + 0.02, f'{value:.2f}', color='black', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 1. Generate "Out-of-Fold" predictions and probabilities for every customer
# This ensures that for every prediction, the model was NOT trained on that specific row
y_pred = cross_val_predict(pipeline, X, y, cv=skf, n_jobs=-1)
y_scores = cross_val_predict(pipeline, X, y, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]

# 2. Compute the Matrix
cm = confusion_matrix(y, y_pred)

# 3. Visualize
plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stay', 'Churn'])
disp.plot(cmap='Blues', values_format='d')

plt.title('Consolidated Confusion Matrix (5-Fold CV Results)', fontsize=15)
plt.grid(False) # Clean up for publication
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Extract and Average Coefficients safely across 5 folds
# .flatten() ensures we handle the (1, N) shape of LogisticRegression.coef_ correctly
# We use a list comprehension to ensure we only grab valid 1D arrays
coef_list = [est.named_steps['classifier'].coef_.flatten() for est in cv_results['estimator']]

# 2. Convert to a 2D array (5 folds x N features)
# This will now work because all arrays are flat 1D sequences
coef_array = np.vstack(coef_list)
mean_coefs = np.mean(coef_array, axis=0)

# 3. Get cleaned feature names from the first fitted pipeline
feature_names = cv_results['estimator'][0].named_steps['preprocessor'].get_feature_names_out()
clean_names = [f.replace('num__', '').replace('cat__', '') for f in feature_names]

# 4. Create DataFrame and Sort for visualization
feat_imp = pd.DataFrame({
    'Feature': clean_names,
    'Coefficient': mean_coefs
}).sort_values(by='Coefficient', ascending=False)

# 5. Generate the Final Visualization for the Presentation
plt.figure(figsize=(10, 8))

# Color coding: Red for Churn Drivers (+), Green for Retention Anchors (-)
colors = ['#253f4b' if x > 0 else '#c8dfea' for x in feat_imp['Coefficient']]

sns.barplot(data=feat_imp, x='Coefficient', y='Feature', hue='Feature', palette=colors, legend=False)

plt.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
plt.title('Predicting "Confirmed Churn": Mean Impact (5-Fold CV)', fontsize=15, pad=20)
plt.xlabel('Strength of Impact on Churn (Log-Odds)', fontsize=12)
plt.ylabel('Operational & Economic Features', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Initialize the Random Forest with 'Bricolage' constraints
# 'class_weight=balanced' is crucial because your churn target is a minority class
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # Use your existing preprocessor
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1 # Uses all CPU cores for speed
    ))
])

# 2. Run Stratified 5-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


scoring_metrics = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results_rf = cross_validate(
    rf_pipeline, X, y,
    cv=skf,
    scoring=scoring_metrics,
    return_estimator=True,
    n_jobs=-1
)



# 3. Create a Detailed Performance Table
cv_df = pd.DataFrame(cv_results_rf)

# Rename columns from cv_results keys to desired display names
cv_df = cv_df.rename(columns={
    'fit_time': 'Fit Time',
    'score_time': 'Score Time',
    'test_accuracy': 'Accuracy',
    'test_precision': 'Precision',
    'test_recall': 'Recall',
    'test_f1': 'F1 Score',
    'test_roc_auc': 'ROC-AUC'
})

# Set index
cv_df.index = [f'Fold {i+1}' for i in range(len(cv_df))]

# Metrics for which we want mean/std
metrics_to_summarize = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'Fit Time', 'Score Time']

summary_stats_data = {}
for metric in metrics_to_summarize:
    summary_stats_data[metric] = [cv_df[metric].mean(), cv_df[metric].std()]

summary_stats = pd.DataFrame(summary_stats_data, index=['MEAN', 'STD DEV'])

# Select desired columns from cv_df for final report (excluding 'estimator')
display_cols = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'Fit Time', 'Score Time']
final_report_table = pd.concat([cv_df[display_cols], summary_stats])

print(final_report_table.round(4))

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

# 1. Generate "Out-of-Fold" predictions for the ENTIRE dataset
# This ensures every row's prediction comes from a model that didn't train on it.
y_probas = cross_val_predict(rf_pipeline, X, y, cv=skf, method='predict_proba')
y_pred = cross_val_predict(rf_pipeline, X, y, cv=skf)

# 2. Setup the Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Graph A: The K-Fold Confusion Matrix (Total Dataset)
cm = confusion_matrix(y, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stay', 'Churn']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Aggregated K-Fold Confusion Matrix')

# Graph B: The K-Fold ROC Curve
fpr, tpr, thresholds = roc_curve(y, y_probas[:, 1])
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', label=f'Mean ROC-AUC = {roc_auc:.2f}')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_title('Aggregated K-Fold ROC Curve')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Extract feature names from the first estimator's preprocessor
feature_names = cv_results_rf['estimator'][0].named_steps['preprocessor'].get_feature_names_out()
clean_names = [f.replace('num__', '').replace('cat__', '') for f in feature_names]

# 2. Extract importances from all 5 models and average them
all_importances = [est.named_steps['classifier'].feature_importances_ for est in cv_results_rf['estimator']]
mean_importances = np.mean(all_importances, axis=0)
std_importances = np.std(all_importances, axis=0) # To show stability/error bars

# 3. Create a DataFrame for plotting
importance_df = pd.DataFrame({
    'Feature': clean_names,
    'Importance': mean_importances,
    'Std Dev': std_importances
}).sort_values(by='Importance', ascending=False)

# 4. Plotting the top 10
plt.figure(figsize=(10, 7))
sns.barplot(
    data=importance_df.head(10),
    x='Importance',
    y='Feature',
    palette='magma'
)

# Optional: Add error bars to show how consistent the importance was across folds
plt.errorbar(
    x=importance_df.head(10)['Importance'],
    y=range(10),
    xerr=importance_df.head(10)['Std Dev'],
    fmt='none', c='black', capsize=3
)

plt.title('Top 10 Churn Drivers (Consensus across 5 Folds)', fontsize=14)
plt.xlabel('Mean Feature Importance Score')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()